In [52]:
import pandas as pd
#from pandas.io.parsers import ParserError
import numpy as np
from helper import get_mapper
import json
import os
import re

In [53]:
from os import listdir, stat
from os.path import isfile, join
BASE_DIR = "."
MIN_SIZE = 512

In [54]:
def extract_blockidf(fullname):
    return fullname.split("Generation_DE ")[1].rsplit('[MW]')[0].strip()

def extract_blockidf2(fullname, plantname):
    return fullname.split("Generation_DE ")[1].rsplit('[MW]')[0].strip().removeprefix(plantname + " ")

def get_smard_name(f):
    return f.rsplit("/")[2].rsplit("_", 4)[0].strip()

def get_smard_name_win(f):
    f = f.replace("\\","/", 2)
    return f.rsplit("/")[2].rsplit("_", 4)[0].strip()

In [55]:
get_smard_name_win("'.\\2015/Abwinden-Asten_201501010000_201512312359_Stunde_1.csv'")

'Abwinden-Asten'

In [56]:
get_smard_name("./2015/Abwinden-Asten_201501010000_201512312359_hour_1.csv")

'Abwinden-Asten'

In [57]:
def get_files_from_folder(folder):
    onlyfiles = [folder + "/" + f for f in listdir(folder) if isfile(join(folder, f))]
    onlyfiles.sort()
    files = [f for f in onlyfiles if stat(f).st_size > MIN_SIZE]
    return files

In [58]:
def convert2plantid(df, plantname):
    oldcols = list(df.columns)
    newcols = [extract_blockidf2(x, plantname) for x in list(df.columns)[1:]]
    #print(plantname + ":" + str(newcols))
    try:
        newcols2 = ['produced_at'] + [mapper[plantname][x] for x in newcols]
    except KeyError:
        newcols2 = ['produced_at'] + [mapper[plantname][x.split(" ")[-1]] for x in newcols]
    test = dict(zip(oldcols, newcols2))
    result = df.rename(columns=test)
    #print(oldcols)
    #print(newcols2)
    return result

In [59]:
def get_plant_from_prod_name(prodname):
    tmp = ""
    try:
        tmp = mapper[prodname]
    except KeyError:
        #print("plant " + prodname + " not found!")
        return None, None
    blocklist = tmp['list']
    block = blocklist[0]
    blockid = tmp[block]
    
    try:
        plantidx = bpm.loc[bpm.blockid == blockid, 'plantid'].item()
    except ValueError:
        return blockid, np.nan
    
    return blockid, plantidx

In [60]:
def get_plant_from_prod_name2(prodname):
    tmp = ""
    try:
        tmp = newmapper[prodname]
    except KeyError:
        #print("plant " + prodname + " not found!")
        return None, None
    seelist = tmp['list']
    block = seelist[0]
    blockid = tmp[block]
    
    try:
        plantidx = seem.loc[seem.sseid == blockid, 'plantid'].item()
    except ValueError:
        return blockid, np.nan
    
    return blockid, plantidx

In [61]:
#bpm = pd.read_csv("../basic/block_plant_mapper.csv")
raw_sse = pd.read_csv("../basic/inspire_prtr_mapper.csv")
see = raw_sse.rename(columns={"InspireID_Betrieb": "plantid"})
seem = see[['plantid', 'sseid']]
#bpm['plantid'] = bpm['plantid'].apply(lambda x: str(x).replace('/', '_'))
seem['plantid'] = seem['plantid'].apply(lambda x: str(x).replace('/', '_'))

/tmp/ipykernel_1700930/2092039524.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  seem['plantid'] = seem['plantid'].apply(lambda x: str(x).replace('/', '_'))


In [62]:
FOLDERS = [os.path.join(BASE_DIR, o) for o in os.listdir(BASE_DIR) if os.path.isdir(os.path.join(BASE_DIR,o))]
FOLDERS.sort()
FOLDERS = FOLDERS[1:-4]
#FOLDERS = FOLDERS[0:1]

In [63]:
FOLDERS

['./2015',
 './2016',
 './2017',
 './2018',
 './2019',
 './2020',
 './2021',
 './2022',
 './2023',
 './2024']

In [64]:
FILES_L = [get_files_from_folder(f) for f in FOLDERS]
FILES = [item for sublist in FILES_L for item in sublist]

In [65]:
#FILES

In [66]:
#mapper = get_mapper('../production/plantmapper.json')
mapper = get_mapper('newmapper.json')
newmapper = get_mapper('newmapper.json')

In [67]:
#get_plant_from_prod_name("Buschhaus")

In [68]:
get_plant_from_prod_name2("Datteln")

('BNA0189', 'NW500-0915123')

In [69]:
#mapper

In [70]:
seem

,plantid,sseid
0,06-02-B10117A007,SEE987197130805
1,06-02-B10117A007,SEE913896693631
2,06-05-100-0030723,BNA1084
3,06-05-100-0431554,BNA0992
4,06-05-100-0431554,BNA0991
...,...,...
887,TH72012874,SEE912583238223
888,TH72012874,SEE969004747543
889,TH72012874,SEE951160693116
890,TH86012942,SEE925388069183


In [71]:
#FILES

In [72]:
#FILES

In [73]:
def get_year(fn):
    return int(fn.split("/")[1])

In [74]:
def get_year2(fn):
    return int(fn.rsplit("_", 3)[1][0:4])

In [75]:
get_year2("./2015/Boxberg_201501010000_201512312345_Stunde_71.csv")

2015

In [76]:
FILES[1]

'./2015/Altenw_rth_201501010000_201512312359_Stunde_3.csv'

In [77]:
for f in FILES:
    #print(f[1])
    fn = get_smard_name_win(f)
    #print(fn)

In [78]:
prod_mapper = []
for f in FILES:
    fn = get_smard_name_win(f)
    f = f.replace("\\","/", 2)
    #print(fn)
    blockid, plantid = get_plant_from_prod_name2(fn)
    #print(fn or "" + ":" + blockid or "" + "->" + plantid or "")
    year = 2015
    try:
        year = get_year2(f)
    except IndexError:
        print(fn)
        pass
    prod_mapper.append([f, blockid, plantid, year])

In [79]:
date_format={'Datum von': '%d-%m-%Y %H:%M'}

In [80]:
df = pd.read_csv("./2024/Heizkraftwerk_Lausward_202401010000_202412312359_Stunde_61.csv", delimiter=";", na_values=['-'], thousands='.', decimal=',', parse_dates=["Datum von"], date_format={'Datum von': '%d-%m-%Y %H:%M'}, on_bad_lines='skip')

In [81]:
#df.dtypes

In [82]:
df.fillna(0, inplace=True)
df[df.columns[2:]] = df[df.columns[2:]].astype(int)

In [83]:
df[df.columns[2:]] = df[df.columns[2:]].astype(int)

In [84]:
prd_df = pd.DataFrame(prod_mapper, columns = ['file', 'blockid', 'plantid', 'year'])

In [85]:
dedup = prd_df.dropna(subset=['blockid', 'plantid'])

In [86]:
for index, row in list(dedup.iterrows()):
    dfname = row.iloc[0]
    plantid = str(row.iloc[2])
    year = str(int(row.iloc[3]))
    smardname = get_smard_name(dfname)
    try:
        df = pd.read_csv(dfname, delimiter=";", na_values=['-'], thousands='.', decimal=',', parse_dates=["Datum von"], on_bad_lines='skip')
        df['Datum von'] = pd.to_datetime(df['Datum von'], format='%d.%m.%Y %H:%M')
        df = df.drop('Datum bis', axis=1)
    except ValueError:
        df = pd.read_csv(dfname, delimiter=";", na_values=['-'], thousands='.', decimal=',', parse_dates=["Datum von"], on_bad_lines='skip')
        df = df.drop('Datum bis', axis=1)
    try:
    #print(smardname)
        newdf = convert2plantid(df, smardname)
        newdf.fillna(0, inplace=True)
        newdf[newdf.columns[1:]] = newdf[newdf.columns[1:]].astype(int)
        newdf.to_csv("./by_plantid/" + year + "/" + plantid + '.csv', index=False)
    except (IndexError, KeyError, ValueError):
        print(smardname)
        continue
    
    #try:
    #print(dfname, year)
    #print(type(year))
    
    #except TypeError:
    #    print(year, smardname)
    #print(newdf.dtypes)

Kraftwerk_BASF_Ludwigshafen_Mitte


/tmp/ipykernel_1700930/3021680282.py:7: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df = pd.read_csv(dfname, delimiter=";", na_values=['-'], thousands='.', decimal=',', parse_dates=["Datum von"], on_bad_lines='skip')
/tmp/ipykernel_1700930/3021680282.py:11: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df = pd.read_csv(dfname, delimiter=";", na_values=['-'], thousands='.', decimal=',', parse_dates=["Datum von"], on_bad_lines='skip')


Lichterfelde
Zolling
Kraftwerk_BASF_Ludwigshafen_Mitte


/tmp/ipykernel_1700930/3021680282.py:7: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df = pd.read_csv(dfname, delimiter=";", na_values=['-'], thousands='.', decimal=',', parse_dates=["Datum von"], on_bad_lines='skip')
/tmp/ipykernel_1700930/3021680282.py:11: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df = pd.read_csv(dfname, delimiter=";", na_values=['-'], thousands='.', decimal=',', parse_dates=["Datum von"], on_bad_lines='skip')


Lichterfelde
Zolling
Kraftwerk_BASF_Ludwigshafen_Mitte


...

Zolling
Kraftwerk_BASF_Ludwigshafen_Mitte
Lichterfelde


/tmp/ipykernel_1700930/3021680282.py:7: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df = pd.read_csv(dfname, delimiter=";", na_values=['-'], thousands='.', decimal=',', parse_dates=["Datum von"], on_bad_lines='skip')
/tmp/ipykernel_1700930/3021680282.py:11: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df = pd.read_csv(dfname, delimiter=";", na_values=['-'], thousands='.', decimal=',', parse_dates=["Datum von"], on_bad_lines='skip')


Zolling
Kraftwerk_BASF_Ludwigshafen_Mitte
Lichterfelde


/tmp/ipykernel_1700930/3021680282.py:7: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df = pd.read_csv(dfname, delimiter=";", na_values=['-'], thousands='.', decimal=',', parse_dates=["Datum von"], on_bad_lines='skip')
/tmp/ipykernel_1700930/3021680282.py:11: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df = pd.read_csv(dfname, delimiter=";", na_values=['-'], thousands='.', decimal=',', parse_dates=["Datum von"], on_bad_lines='skip')


Zolling


In [87]:
extract_blockidf('Generation_DE Bergkamen A [MW] Originalauflösungen\n')

'Bergkamen A'